In [ ]:
import time

import torch
from torch import nn

from extra.Guia10Reformat.src.Config import Config
from extra.Guia10Reformat.src.FashionMNISTDATA import FashionMNISTData
from src.Autoencoder import Autoencoder
from src.Optimizer import create_optimizer
from src.Trainer import Trainer
from src.Visualizer import Visualizer


def main(config):
    # Configuración
    device = config.setup()
    # Datos
    data = FashionMNISTData(
        batch_size=config.BATCH_SIZE,
        seed=config.SEED, color=config.COLOR
    )

    if config.showSample:
        data.inspect_example()
        data.show_samples()

    # Modelo
    torch.manual_seed(Config.SEED)

    model = Autoencoder(
        latent_dim=config.LATENT_DIM,
        p_dropout=config.DROPOUT_P
    ).to(device)

    # Comprobar dimensiones
    if config.showDimentions:
        images, labels = next(iter(data.train_loader))

        with torch.no_grad():
            reconstructions = model(images.to(device))
            latent_codes = model.encode(images.to(device))

        print("\nDimensiones:")
        print("Entrada:", images.shape)
        print("Código latente:", latent_codes.shape)
        print("Reconstrucción:", reconstructions.shape)

    # Función de pérdida y optimizador
    loss_fn = nn.MSELoss()

    optimizer = create_optimizer(
        model,
        config.OPTIMIZER_NAME
    )

    print("\nOptimizador:", config.OPTIMIZER_NAME)
    print("Dropout:", config.DROPOUT_P)

    # Reconstrucciones antes del entrenamiento
    if config.showUntrained:
        Visualizer.show_reconstructions(
            model,
            data.valid_loader,
            device,
            data.CLASS_NAMES,
            title="Sin entrenar", color=config.COLOR
        )

    # Entrenamiento
    trainer = Trainer(
        model,
        optimizer,
        loss_fn,
        device
    )
    start = time.perf_counter()
    trainer.fit(
        data.train_loader,
        data.valid_loader,
        epochs=config.EPOCHS
    )

    print(f"Training time: {time.perf_counter() - start:.2f} seconds")
    # Gráfico de pérdidas
    if config.showLoss: Visualizer.plot_history(trainer.history)

    # Evaluación final
    trainer.test(data.test_loader)

    # Reconstrucciones después del entrenamiento
    if config.showTrained: Visualizer.show_reconstructions(
        model,
        data.test_loader,
        device,
        data.CLASS_NAMES,
        title="Reconstrucción", color=config.COLOR
    )

    # Espacio latente
    if config.showDimentions2:
        images, _ = next(iter(data.test_loader))

        images = images.to(device)

        model.eval()

        with torch.no_grad():
            latent_codes = model.encode(images)

        print("\nEspacio latente:")
        print("Imágenes:", images.shape)
        print("Códigos latentes:", latent_codes.shape)


# ============================================================
# EJECUCIÓN
# ============================================================

if __name__ == "__main__":
    config1 = Config(latent_dim=16, dropout_p=0.1, showSample=False, showDimentions=False, showUntrained=False,
                     showLoss=False, showTrained=True, showDimentions2=False)

    config1 = Config(latent_dim=4, dropout_p=0.1, showSample=False, showDimentions=False, showUntrained=False,
                     showLoss=False, showTrained=True, showDimentions2=False)

    toRun = [config1]
    for config in toRun:
        main(config)



========== CONFIGURATION ==========
Seed:             42
Batch size:       100
Epochs:           10
Latent dimension: 4
Dropout:          0.1
Optimizer:        Adam

========== ENVIRONMENT ==========
PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


Cantidad de imágenes:
Train: 50000
Validation: 10000
Test: 10000

Optimizador: Adam
Dropout: 0.1

Iniciando entrenamiento...
Dispositivo: cuda
Época 01/10 | Train MSE: 0.07416 | Valid MSE: 0.07444
Época 02/10 | Train MSE: 0.06160 | Valid MSE: 0.06190
Época 03/10 | Train MSE: 0.05888 | Valid MSE: 0.05919
Época 04/10 | Train MSE: 0.05468 | Valid MSE: 0.05504
Época 05/10 | Train MSE: 0.04932 | Valid MSE: 0.04968
Época 06/10 | Train MSE: 0.04662 | Valid MSE: 0.04696
